# Week 11 — Active Follow-up, Bayesian Optimisation, Scaling and Scientific Synthesis

**Contact:** 3 hours lecture  
**Course:** MSc Astrostatistics for the Survey Era

## Learning notebooks from the Crete school
- `../../09_Bayesian_Optimization/Bayesian_Optimization.ipynb`
- `../../13_GPU_parallelization/GPU_parallelization_answerkey.ipynb`
- `../../14_SBI/SBI_exercise.ipynb`

These Crete notebooks are **learning material**. This lecture notebook supplies the semester narrative, deeper astronomical interpretation, and the binary-SMBH case study.

## Three-hour pacing
- **0:00–0:20** — recap, motivating scientific question, diagnostic poll
- **0:20–1:05** — theory block I + worked derivation
- **1:05–1:15** — short break / discussion
- **1:15–2:00** — theory block II + computational demonstration
- **2:00–2:10** — short break
- **2:10–2:45** — binary-SMBH case study / failure mode
- **2:45–3:00** — synthesis, exit questions, bridge to the 4-hour practical

## Learning objectives

- Formulate follow-up selection as a utility/decision problem.
- Explain exploration versus exploitation in Bayesian optimisation.
- Use expected information gain as a conceptual observation utility.
- Design scalable Parquet workflows without loading the full simulation store.
- Synthesise the full course into a defensible end-to-end scientific inference pipeline.

## 1. From inference to decisions


Astronomers rarely have unlimited follow-up resources. Once an inference model identifies candidates, the next question is which observation would be most valuable.

Decision theory separates uncertain states from actions and utilities. If $a$ is an action and $\theta$ the unknown state, choose

$$
a^\ast=\arg\max_a E[U(a,\theta)\mid D].
$$

Utility can represent classification yield, expected precision, scientific novelty, observing cost or combinations of these. The definition must be explicit because different utilities produce different target lists.

> **Discussion checkpoint.** Identify the estimand, observed data, and strongest modelling assumption in the example just discussed.

### Worked computational demonstration — Simple expected-improvement demonstration

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
import numpy as np
from scipy.stats import norm

def expected_improvement(mu,sigma,best,xi=0.01):
    sigma=np.maximum(np.asarray(sigma),1e-12)
    imp=np.asarray(mu)-best-xi
    z=imp/sigma
    return imp*norm.cdf(z)+sigma*norm.pdf(z)

mu=np.array([.50,.55,.48])
sig=np.array([.02,.15,.30])
print(expected_improvement(mu,sig,best=.52))

## 2. Expected information gain


One principled utility is expected information gain: choose the observation $y$ that is expected to reduce uncertainty about $\theta$. Conceptually,

$$
{\rm EIG}(a)
=
E_{y\sim p(y\mid a,D)}
\left[
{\rm KL}\big(p(\theta\mid D,y,a)\,\|\,p(\theta\mid D)\big)
\right].
$$

Exact calculation can be expensive, but the formula clarifies what "informative follow-up" means. A high predicted probability alone is not the same as high information gain; an object near a decision boundary can be more informative.

### Worked computational demonstration — Parquet projection pattern

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
# This cell is illustrative; point it to the private large dataset.
# import pyarrow.parquet as pq
# pf=pq.ParquetFile('../data/private/tenpct_dataset_withinputs.parquet')
# table=pf.read_row_group(0, columns=[
#     "('inputs', 'mass ratio')",
#     "('inputs', 'period')",
# ])
# print(table.num_rows, table.num_columns)

## 3. Bayesian optimisation


Bayesian optimisation places a surrogate model, often a GP, over an expensive objective $f(x)$. An acquisition function balances exploration of uncertain regions and exploitation of promising regions.

Expected improvement, upper confidence bound and probability of improvement encode different trade-offs. In astronomy the "objective" might be a simulation score, observing utility or expensive likelihood computation rather than a black-box industrial function.

### Worked computational demonstration — Reproducibility manifest pattern

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
import json, platform
manifest={
    'python':platform.python_version(),
    'random_seed':2026,
    'target':['mass_ratio','log10_period'],
    'split':'document exact rule here',
    'notes':'record dataset/extraction version here'
}
print(json.dumps(manifest,indent=2))

## 4. Scaling: data layout before hardware


The binary-SMBH parent dataset is hundreds of GB because time, magnitude and error arrays dominate storage. The scalar physical inputs occupy a tiny fraction. This is a textbook reason to use Parquet projection: read only required columns and row groups.

Efficient workflows use:

- schema/metadata inspection before scanning;
- column projection;
- row-group or batch processing;
- vectorisation;
- compact derived feature tables;
- GPUs only when the computational kernel benefits.

A GPU does not rescue an algorithm that needlessly reads hundreds of GB.

## 5. Reproducibility and provenance


A research-grade analysis records dataset version, extraction commands, random seeds, train/test split logic, environment, model configuration and all transformations. Simulation analyses should also record the simulation prior and mechanism switches.

The final project is deliberately structured so that another student should be able to reproduce the validation results from the repository, while the blind truth remains private.

## 6. Scientific synthesis: the full loop


By Week 11, students should be able to articulate the full chain:

$$
\text{population assumptions}
\rightarrow
\text{latent SMBH parameters}
\rightarrow
\text{variability physics}
\rightarrow
\text{survey cadence/noise}
\rightarrow
\text{selected catalogue}
\rightarrow
\text{inference}
\rightarrow
\text{validation}
\rightarrow
\text{follow-up decision}.
$$

The purpose of the final project is not to choose the fanciest model. It is to show that the student can defend every arrow in this diagram and knows where the inference could fail.

## 7. Final-project scientific defence


The final oral presentation should prioritise scientific validity. Students should show one figure for predictive performance, one for uncertainty calibration, one stress-test/domain-shift result, and one example of a failure case.

A strong conclusion may be that one target parameter is poorly identifiable from the supplied observables. Demonstrating this convincingly is better science than hiding the limitation behind a complex model.

## Deep dive — observation design as Bayesian decision theory

Suppose a candidate has a posterior over physical parameters. Different possible follow-up measurements $a$ imply different predictive distributions $p(y\mid a,D)$. After observing $y$, the posterior changes. Expected utility averages over possible future outcomes.

This turns target selection into a nested inference problem. Exact EIG can be computationally demanding, so practical proxies are common: posterior entropy, expected variance reduction, classifier uncertainty, diversity bonuses or simple physically motivated scores.

### Exploration, exploitation and catalogue bias

If follow-up always selects the objects with the largest predicted probability of being interesting, the follow-up sample becomes strongly selected. Future population studies using that sample must model this adaptive selection.

Active learning can therefore improve discovery efficiency while simultaneously making downstream population inference more complex.

### Scaling the scientific workflow

At survey scale, one should separate:
- raw/heavy data;
- compact derived features;
- model artefacts/checkpoints;
- posterior summaries;
- provenance and configuration.

The 500+ GB SMBH store is an excellent teaching example: most course tasks can use a tiny scalar or feature table, while only specific time-domain/SBI stages touch the heavy arrays.

### What a complete scientific claim looks like

A final claim should include:
- target population and estimand;
- data and selection;
- model family;
- validation regime;
- uncertainty/calibration;
- failure region;
- simulation-to-real limitations;
- conditions under which the result should be revised.

That standard is the real final learning outcome of the course.

## Binary-SMBH synthesis questions

Answer verbally before leaving the lecture.

1. Which quantity in today's topic is a **property of the simulator**, and which could be an **observable property of a real survey**?
2. What uncertainty or selection effect would most strongly distort the result?
3. What diagnostic would you require before trusting the inference?
4. What information would be unavailable if these were real LSST sources rather than simulations?

## Key equations / ideas to retain

Create your own 6–10 line summary here during class.

## Further reading

- Crete `09_Bayesian_Optimization`, `13_GPU_parallelization`, `14_SBI`.
- Rasmussen & Williams for GP surrogates.
- Course data-architecture documentation.

# Part II — Extended lecture development

## 15. Decision theory with asymmetric costs

Suppose a false positive costs one unit of telescope time while missing a rare
binary costs ten scientific utility units. The probability threshold minimizing
expected loss is not necessarily 0.5.

For binary state $Y$ and actions $a$, compare posterior expected losses

$$
R(a\mid D)=\sum_y L(a,y)P(y\mid D).
$$

This formalises why catalogue thresholds should reflect programme goals rather
than software defaults.

## 16. Value of information

A future observation has value because it can change decisions. Expected value
of information compares expected utility with and without the observation.

If an object is already classified with near certainty and the action would not
change, more data can have low decision value even if scientifically
interesting.

This separates **uncertainty reduction** from **decision relevance**.

## 17. Diversity-aware follow-up

Selecting only the top score can concentrate observations in one narrow region.
A programme may instead want diversity in redshift, period, mechanism or feature
space.

A utility can include a diversity penalty/bonus. Alternatively, target
selection can be formulated as an optimisation over a set rather than
independent object scores.

This is especially relevant when follow-up data will later calibrate a model:
coverage of parameter space can matter more than immediate yield.

## 18. Active learning and feedback loops

Once follow-up labels are added to training, the model influences which data it
will see next. This creates a feedback loop.

If uncertainty estimates are biased, the active learner can systematically
ignore poorly understood regions.

Logging every selection rule and occasionally allocating exploration targets
helps prevent a self-reinforcing narrow training distribution.

## 19. Computational complexity and scaling laws

A naive GP with $n$ data points requires roughly $O(n^3)$ factorisation and
$O(n^2)$ memory for a dense covariance matrix.

Tree ensembles scale differently; neural inference shifts cost to training and
then makes per-object amortised inference cheap.

Algorithm choice must therefore consider:
- number of objects;
- epochs per object;
- dimensionality;
- number of repeated inferences;
- hardware;
- required uncertainty.

The statistically elegant method may be computationally impossible at survey
scale without approximation.

## 20. Feature-table architecture

For the SMBH course, a good architecture is:

1. heavy raw Parquet with nested light curves;
2. compact scalar truth/context table;
3. compact observable feature table;
4. train/validation/blind splits;
5. model outputs/posterior summaries.

This avoids repeatedly scanning hundreds of GB and makes the statistical
experiment reproducible.

The feature-extraction step should itself be versioned because changing features
changes the scientific model.

## 21. CPU, vectorisation and GPU hierarchy

Before GPU acceleration:
1. avoid repeated file scans;
2. project columns;
3. vectorise NumPy/Pandas operations;
4. batch computations;
5. profile bottlenecks.

Only then move arithmetic-heavy kernels to a GPU.

Data transfer and GPU memory can dominate if the workload is small or I/O-bound.

## 22. Reproducibility versus replicability

**Reproducibility:** the same data/code/environment reproduce the reported
result.

**Replicability:** an independent dataset/experiment supports the scientific
claim.

Simulation exercises can be highly reproducible while having limited
replicability to the real Universe. A mature scientific report states both.

## 23. Final-project scoring philosophy

The common blind project deliberately rewards:
- correct split/leakage control;
- calibrated uncertainty;
- robustness under shift;
- failure analysis;
- reproducible code;
- appropriate scope.

A model with marginally better IID RMSE but severe overconfidence can score
below a simpler calibrated model. This aligns assessment with scientific
reliability rather than leaderboard gaming.

## 24. Final course synthesis: eleven questions

By the end of the course, a student should be able to answer:

1. What is the estimand?
2. What is observed?
3. What is latent?
4. What generates the data?
5. What is the likelihood—or why is it intractable?
6. What uncertainty is measured?
7. What selection occurred?
8. What information leaks are possible?
9. How is calibration checked?
10. Where does support/domain shift break the inference?
11. Which observation should be taken next?

If those questions are answered clearly, the statistical method is usually
much easier to evaluate.

# Part III — Final seminar problems

## Problem A — probability versus utility

Candidate A has 95% probability of being a binary but is already well
characterised. Candidate B has 60% probability but a new spectrum would sharply
separate competing physical models.

Which should be observed?

There is no answer until the utility is stated. A discovery-yield utility may
prefer A; information gain may prefer B.

## Problem B — follow-up creates selection

Suppose only candidates with predicted probability $>0.9$ receive spectra.
Later you estimate the binary fraction using only the spectroscopic sample.

This is biased unless the probability of spectroscopic selection is incorporated
into the population analysis. The decision policy has become part of the survey
selection function.

## Problem C — reproducible but not externally valid

Every student can exactly reproduce a neural posterior trained on simulations,
including random seed and environment.

Does this establish that the posterior is valid for real LSST AGN?

No. That is computational reproducibility. External validity requires evidence
that the simulation-to-real relationship is adequate. The distinction should
appear explicitly in final conclusions.

# Part IV — Additional synthesis seminar

## Scientific utility versus statistical uncertainty

Uncertainty alone is not a utility. An uncertain object can be scientifically
unimportant, while a moderately uncertain rare system can be transformative.

A follow-up programme should document both the posterior state and the value
function used to rank actions.

## Pipeline versioning

At survey scale, a result depends on more than model weights. Version:
- feature extractor;
- data selection;
- input schema;
- normalisation;
- trained model;
- calibration layer;
- decision threshold.

Changing any of these changes the effective inference system.

## Closing principle

The "next frontier" of astrostatistics is not one algorithm. It is the ability
to connect physical simulators, survey selection, probabilistic inference,
uncertainty calibration and active observing into one validated loop.

Students who can explain that loop—and diagnose where it fails—are prepared to
work with Rubin-scale data even as specific ML architectures change.